In [28]:
# Import Libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

In [29]:
# Load Train Data
train_df = pd.read_csv('../data/train.csv')

In [30]:
# Exploratory Data Analysis (EDA)
print("--- Check for Missing Values ---")
print(train_df.isnull().sum())
print("-" * 40)

print("--- Survival Rate by Embarked Port ---")
# Check which port has a higher survival rate
embarked_survival = train_df[['Embarked', 'Survived']].groupby(['Embarked']).mean()
print(embarked_survival)
print("-" * 40)

print("--- Survival Rate by Sex ---")
# Analyze if sex affects the survival rate
sex_survival = train_df[['Sex', 'Survived']].groupby(['Sex']).mean()
print(sex_survival)
print("-" * 40 + "\n")

--- Check for Missing Values ---
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64
----------------------------------------
--- Survival Rate by Embarked Port ---
          Survived
Embarked          
C         0.553571
Q         0.389610
S         0.336957
----------------------------------------
--- Survival Rate by Sex ---
        Survived
Sex             
female  0.742038
male    0.188908
----------------------------------------



In [31]:
# Feature Engineering & Data Cleaning

# Extract 'Title' from the 'Name' column
train_df['Title'] = train_df['Name'].str.split(',').str[1].str.split('.').str[0].str.strip()

# Print to check the extracted titles
print("--- Passenger Titles Extracted ---")
print(train_df['Title'].value_counts())
print("-" * 40 + "\n")

# Fill missing Age values with the median
train_df['Age'] = train_df['Age'].fillna(train_df['Age'].median())

# Fill missing Embarked values with the mode (most frequent port)
train_df['Embarked'] = train_df['Embarked'].fillna(train_df['Embarked'].mode()[0])

# Encode categorical text data into numerical values
train_df['Sex'] = train_df['Sex'].map({'male': 0, 'female': 1})
train_df['Embarked'] = train_df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

# Group rare titles into a single category named 'Rare'
rare_titles = ['Dr', 'Rev', 'Col', 'Major', 'Countess', 'Sir', 'Jonkheer', 'Lady', 'Capt', 'Don', 'Dona']
train_df['Title'] = train_df['Title'].replace(rare_titles, 'Rare')

# Standardize synonymous titles (e.g., French/English variations)
train_df['Title'] = train_df['Title'].replace(['Mlle', 'Ms'], 'Miss')
train_df['Title'] = train_df['Title'].replace('Mme', 'Mrs')

# Encode the text titles into numerical values
title_mapping = {'Mr': 1, 'Miss': 2, 'Mrs': 3, 'Master': 4, 'Rare': 5}
train_df['Title'] = train_df['Title'].map(title_mapping)

# Fill any remaining missing values with 0 (as a safety net)
train_df['Title'] = train_df['Title'].fillna(0)

# Create FamilySize Feature
train_df['FamilySize'] = train_df['SibSp'] + train_df['Parch'] + 1

# Create a function to categorize family size into logical groups
def group_family(size):
    if size == 1:
        return 'Alone'
    elif size <= 4:
        return 'Small'
    else:
        return 'Large'

# Apply the function to create a new 'FamilyGroup' column
train_df['FamilyGroup'] = train_df['FamilySize'].apply(group_family)

# Encode the groups into numbers
family_mapping = {'Small': 1, 'Alone': 2, 'Large': 3}
train_df['FamilyGroup'] = train_df['FamilyGroup'].map(family_mapping)

--- Passenger Titles Extracted ---
Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Col               2
Mlle              2
Major             2
Ms                1
Mme               1
Don               1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64
----------------------------------------



In [32]:
# Feature Selection & Data Split

# Select features (Removed SibSp and Parch, added Title and FamilyGroup)
features = ['Pclass', 'Sex', 'Age', 'Embarked', 'Title', 'FamilyGroup']
X = train_df[features]
y = train_df['Survived']

# Split data into Train (80%) and Validation (20%) sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [33]:
# Model Training & Evaluation

# Initialize the k-NN model
knn_model = KNeighborsClassifier(n_neighbors=5)

# Train the model using the scaled training data!
knn_model.fit(X_train, y_train)

# Make predictions on the scaled validation data!
y_pred = knn_model.predict(X_val)

# Calculate and print accuracy
accuracy = accuracy_score(y_val, y_pred)
print(f"Validation Accuracy: {accuracy * 100:.2f}%")

Validation Accuracy: 81.56%
